# Descubrimiento de Verticales por Clustering Semántico
### Los clusters emergen de los datos — no los definís vos de antemano

**Flujo completo:**
1. Cada paper → vector semántico (embedding de título + abstract + keywords)
2. UMAP reduce dimensionalidad (384D → 10D para clustering, 2D para visualización)
3. HDBSCAN encuentra clusters naturales sin que definas cuántos
4. TF-IDF extrae keywords representativas de cada cluster
5. **Vos** revisás los clusters y les ponés nombre
6. Se exporta un Excel con cada paper etiquetado y un mapa de conocimiento

**Resultado:** verticales que emergen de tu base real, con nombres que vos asignás.

---
**Requisito:** un archivo `.xlsx` exportado desde Scopus con al menos las columnas `Title`, `Abstract`, `Author Keywords` e `Index Keywords`.

## Celda 1 — Instalación (una sola vez, tarda ~2-3 min)

In [ ]:
import subprocess, sys

paquetes = [
    'sentence-transformers',  # Embeddings semánticos
    'umap-learn',             # Reducción dimensional 10D y 2D
    'hdbscan',                # Clustering sin número fijo de clusters
    'scikit-learn',           # TfidfVectorizer + AgglomerativeClustering
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'openpyxl',               # Lectura y escritura de Excel
    'tqdm',                   # Barra de progreso
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + paquetes + ['--quiet'])
print('✅ Dependencias instaladas')

## Celda 2 — Configuración ✏️

**Solo editá esta celda.** Todo lo demás corre sin cambios.

In [ ]:
# ══════════════════════════════════════════════════════
# EDITÁ SOLO ESTAS VARIABLES
# ══════════════════════════════════════════════════════

# Ruta al archivo Excel exportado desde Scopus
RUTA_DATOS  = r'C:\ruta\a\tu\base_scopus.xlsx'

# Ruta donde se guardará el Excel de resultados
RUTA_SALIDA = r'C:\ruta\a\tu\Clusters_Descubiertos.xlsx'

# Nombre de tu institución (aparece en el mapa y en el resumen)
NOMBRE_INSTITUCION = 'Mi Universidad'

# Nombres de columnas en tu Excel de Scopus
# (verificá que coincidan exactamente con los encabezados de tu archivo)
COL_TITULO   = 'Title'
COL_ABSTRACT = 'Abstract'
COL_KW_AUTOR = 'Author Keywords'
COL_KW_INDEX = 'Index Keywords'

# ══════════════════════════════════════════════════════
# PARÁMETROS DE CLUSTERING — podés ajustar después
# ══════════════════════════════════════════════════════

# Tamaño mínimo de cluster (en papers).
# Valores orientativos según tamaño de la base:
#   ~3.000 papers  → MIN_CLUSTER_SIZE = 30
#   ~5.000 papers  → MIN_CLUSTER_SIZE = 50
#  ~10.000 papers  → MIN_CLUSTER_SIZE = 80  (valor por defecto)
#  ~20.000+ papers → MIN_CLUSTER_SIZE = 120
# Subir este valor → menos clusters, más grandes
# Bajarlo          → más clusters, más granulares
MIN_CLUSTER_SIZE = 80

# Stopwords geográficas propias de tu institución
# Agregá nombres de tu país, ciudad, región para evitar que sesgen los clusters
STOPWORDS_GEOGRAFICAS = {
    # Ejemplo — reemplazá por los de tu institución:
    # 'brazil', 'sao paulo', 'rio', 'brasil',
    # 'mexico', 'ciudad de mexico', 'unam',
    # 'chile', 'santiago', 'pontificia',
}

print('✅ Configuración lista')
print(f'   Institución:      {NOMBRE_INSTITUCION}')
print(f'   Datos:            {RUTA_DATOS}')
print(f'   Salida:           {RUTA_SALIDA}')
print(f'   Min cluster size: {MIN_CLUSTER_SIZE}')

## Celda 3 — Cargar datos y construir texto de cada paper

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm.auto import tqdm

df = pd.read_excel(RUTA_DATOS)
print(f'✅ Dataset: {len(df):,} papers, {len(df.columns)} columnas')
print(f'   Columnas disponibles: {list(df.columns)}')

# ── Stopwords generales (académicas + editoriales) ────────────────────
# Estas palabras son tan genéricas que contaminan la representación semántica.
# NO toques este bloque; editá STOPWORDS_GEOGRAFICAS en la Celda 2.
STOPWORDS_GENERALES = {
    # Relleno académico
    'data', 'model', 'models', 'study', 'results', 'analysis', 'based',
    'different', 'new', 'high', 'method', 'methods', 'properties', 'type',
    'presented', 'proposed', 'use', 'time', 'number', 'structure', 'using',
    'show', 'showed', 'used', 'non', 'two', 'three', 'also', 'well',
    'present', 'work', 'approach', 'paper', 'review', 'found', 'provide',
    'including', 'large', 'small', 'low', 'effect', 'effects', 'role',
    'function', 'functions', 'form', 'forms', 'related', 'observed',
    'experimental', 'theoretical', 'numerical', 'computational',
    'response', 'state', 'states', 'factor',
    # Biología genérica
    'human', 'animal', 'activity', 'expression', 'cell', 'cells',
    'protein', 'proteins', 'gene', 'disease', 'acid', 'binding', 'molecular',
    # Boilerplate de editoriales
    'elsevier', 'reserved', 'rights reserved', 'author', 'article',
    'rights', 'copyright', 'formula', 'formula presented',
    'american physical', 'physical society', 'american physical society',
    'american', 'society', 'editorial', 'springer', 'wiley', 'published',
    # Números y romanos frecuentes
    '13', '19', 'ii', 'iii', 'iv',
}

# Combinar stopwords generales + geográficas de la configuración
STOPWORDS_PRE_EMBEDDING = STOPWORDS_GENERALES | {w.lower() for w in STOPWORDS_GEOGRAFICAS}

patron_stopwords = re.compile(
    r'\b(' + '|'.join(re.escape(w) for w in STOPWORDS_PRE_EMBEDDING) + r')\b',
    re.IGNORECASE
)

def limpiar_texto(texto):
    """Remueve stopwords y normaliza espacios."""
    if not texto or str(texto).lower() in ('nan', ''):
        return ''
    texto = str(texto)
    texto = patron_stopwords.sub(' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def construir_texto(fila):
    """
    Combina campos textuales del paper con ponderación:
    título x2, abstract x3, keywords x2.
    """
    partes = []
    titulo = limpiar_texto(fila.get(COL_TITULO, ''))
    if titulo:
        partes.extend([titulo] * 2)
    abstract = limpiar_texto(fila.get(COL_ABSTRACT, ''))
    if abstract and abstract.lower() != '[no abstract available]':
        partes.extend([abstract] * 3)
    for col in [COL_KW_AUTOR, COL_KW_INDEX]:
        kw = limpiar_texto(fila.get(col, ''))
        if kw:
            partes.extend([kw.replace(';', ',')] * 2)
    return ' '.join(partes)

tqdm.pandas(desc='Construyendo textos')
df['_texto'] = df.progress_apply(construir_texto, axis=1)

n_vacios = (df['_texto'].str.strip() == '').sum()
print(f'✅ Textos construidos. Papers con texto vacío: {n_vacios:,}')
if n_vacios > len(df) * 0.1:
    print('⚠️  Más del 10% de papers sin texto. Verificá que los nombres de columna en la Celda 2 sean correctos.')

## Celda 4 — Calcular embeddings semánticos (~10-15 min en CPU)

Si ya corriste esta celda antes, podés saltearla y usar la **Celda 4b** para cargar desde archivo.

In [ ]:
from sentence_transformers import SentenceTransformer

print('⏳ Cargando modelo (descarga ~80MB la primera vez)...')
modelo = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Modelo listo')

print(f'⏳ Calculando embeddings de {len(df):,} papers...')
embeddings = modelo.encode(
    df['_texto'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(f'✅ Embeddings calculados: {embeddings.shape}')

# Guardar para no recalcular en futuras corridas
np.save('embeddings_papers.npy', embeddings)
print('✅ Embeddings guardados en embeddings_papers.npy')

## Celda 4b — Cargar embeddings ya calculados (opcional)

Descomentá y corrés esta celda **en lugar de la Celda 4** si ya calculaste los embeddings antes.

In [ ]:
# Descomentá las líneas de abajo para cargar desde archivo:
# embeddings = np.load('embeddings_papers.npy')
# print(f'✅ Embeddings cargados: {embeddings.shape}')

## Celda 5 — Reducir dimensionalidad (UMAP) y clustering (HDBSCAN)

Esta celda es la que podés re-correr rápido (~1-2 min) si ajustás `MIN_CLUSTER_SIZE` en la Celda 2.

| `MIN_CLUSTER_SIZE` | Resultado |
|---|---|
| 30-50 | Más clusters, más granulares |
| 80 | Valor por defecto |
| 120-150 | Menos clusters, más amplios |

In [ ]:
import umap
import hdbscan

# ── Paso 1: UMAP 10D para clustering ──────────────────────────────────
print('⏳ UMAP 10D para clustering...')
reducer_10d = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)
emb_10d = reducer_10d.fit_transform(embeddings)
print(f'✅ UMAP 10D: {emb_10d.shape}')

# ── Paso 2: UMAP 2D para visualización ────────────────────────────────
print('⏳ UMAP 2D para visualización...')
reducer_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
emb_2d = reducer_2d.fit_transform(embeddings)
df['_umap_x'] = emb_2d[:, 0]
df['_umap_y'] = emb_2d[:, 1]
print('✅ UMAP 2D listo')

# ── Paso 3: HDBSCAN ───────────────────────────────────────────────────
print(f'⏳ HDBSCAN clustering (min_cluster_size={MIN_CLUSTER_SIZE})...')
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)
etiquetas = clusterer.fit_predict(emb_10d)
df['_cluster_fino'] = etiquetas

n_clusters = len(set(etiquetas)) - (1 if -1 in etiquetas else 0)
n_ruido    = (etiquetas == -1).sum()
print(f'\n✅ Clustering completado:')
print(f'   Clusters encontrados:          {n_clusters}')
print(f'   Papers sin cluster (ruido):    {n_ruido:,} ({100*n_ruido/len(df):.1f}%)')
print(f'\n   Tamaño de cada cluster:')
for c, n in sorted(pd.Series(etiquetas[etiquetas >= 0]).value_counts().items()):
    print(f'   Cluster {c:3d}: {n:,} papers')

## Celda 6 — Extraer keywords de cada cluster

Esta celda genera las keywords que te ayudarán a ponerle nombre a cada cluster en la siguiente.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

# Reutilizamos las mismas stopwords de la Celda 3
STOPWORDS_TFIDF = STOPWORDS_PRE_EMBEDDING

def extraer_keywords_cluster(textos, top_n=20):
    if len(textos) < 3:
        return []
    try:
        vec = TfidfVectorizer(
            max_features=500,
            stop_words='english',
            ngram_range=(1, 2),
            min_df=2
        )
        matriz = vec.fit_transform(textos)
        scores = np.asarray(matriz.mean(axis=0)).flatten()
        vocab = vec.get_feature_names_out()
        indices_filtrados = [
            i for i in scores.argsort()[::-1]
            if vocab[i] not in STOPWORDS_TFIDF
            and not vocab[i].isdigit()
            and len(vocab[i]) > 2
        ][:top_n]
        return [vocab[i] for i in indices_filtrados]
    except:
        return []

info_clusters = {}
for cluster_id in sorted(set(etiquetas)):
    if cluster_id == -1:
        continue
    mask = df['_cluster_fino'] == cluster_id
    textos_cluster  = df[mask]['_texto'].tolist()
    titulos_muestra = df[mask][COL_TITULO].dropna().head(50).tolist()
    keywords = extraer_keywords_cluster(textos_cluster)
    info_clusters[cluster_id] = {
        'n_papers':       mask.sum(),
        'keywords':       keywords,
        'titulos_muestra': titulos_muestra
    }

print(f'✅ Keywords extraídas para {len(info_clusters)} clusters')
print('\nResumen:')
for cid, info in info_clusters.items():
    print(f'\nCluster {cid} ({info["n_papers"]:,} papers):')
    print(f'  Keywords: {" | ".join(info["keywords"][:15])}')
    print(f'  Ejemplo:  {info["titulos_muestra"][0][:90] if info["titulos_muestra"] else ""}')

## Celda 7 — Nombrar los clusters ✏️

**Este es el único paso manual del pipeline.**

1. Corré la celda → se imprime la info de cada cluster (keywords + títulos de ejemplo)
2. Leé los outputs y completá el diccionario `NOMBRES_MANUALES` debajo
3. Volvé a correr la celda para aplicar los nombres

**Formato de cada entrada:**
```python
ID: {'nombre_corto': 'Nombre descriptivo del área', 'vertical_industrial': 'Vertical A / Vertical B'},
```

**Verticales industriales sugeridas** (usá las que apliquen a tu institución):
`AI & ML`, `Biotech`, `Pharma & Drug Discovery`, `Diagnóstico & MedTech`, `Neurotecnología`,
`Energía & Cleantech`, `Agritech`, `Foodtech`, `Materiales Avanzados`, `Quantum Tech`,
`Space Tech`, `Climate Tech`, `Geociencias & Minería`, `Conservación & Ecotech`,
`Social Tech`, `HPC & Big Data`, `Matemática Aplicada`

In [ ]:
# ── Paso 1: imprimí la info de cada cluster para leer antes de nombrar ──
print('=' * 70)
print('REVISÁ LOS CLUSTERS Y COMPLETÁ EL DICCIONARIO NOMBRES_MANUALES ABAJO')
print('=' * 70)
for cid, info in info_clusters.items():
    print(f'\n--- CLUSTER {cid} ({info["n_papers"]:,} papers) ---')
    print(f'Keywords: {" | ".join(info["keywords"][:25])}')
    for t in info['titulos_muestra'][:5]:
        print(f'  · {t[:90]}')

# ── Paso 2: completá este diccionario con los IDs que aparecieron arriba ──
# Copiá el bloque de abajo y agregá una línea por cada cluster.
# Podés usar '/' para asignar más de una vertical a un cluster.

NOMBRES_MANUALES = {
    # 0:  {'nombre_corto': 'Ejemplo: Machine Learning & NLP',  'vertical_industrial': 'AI & ML'},
    # 1:  {'nombre_corto': 'Ejemplo: Oncología Molecular',     'vertical_industrial': 'Pharma & Drug Discovery / Biotech'},
    # ... completá un ID por cada cluster que apareció arriba
}

# ── Paso 3: aplicar los nombres ───────────────────────────────────────
if NOMBRES_MANUALES:
    for cid, nombres in NOMBRES_MANUALES.items():
        if cid in info_clusters:
            info_clusters[cid].update(nombres)
    print(f'\n✅ Nombres aplicados a {len(NOMBRES_MANUALES)} clusters')
    clusters_sin_nombre = [cid for cid in info_clusters if 'nombre_corto' not in info_clusters[cid]]
    if clusters_sin_nombre:
        print(f'⚠️  Clusters sin nombre aún: {clusters_sin_nombre}')
else:
    print('\n⚠️  NOMBRES_MANUALES está vacío. Completá el diccionario y volvé a correr esta celda.')

## Celda 8 — Visualización: Mapa de Conocimiento 2D

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Crear columnas legibles en el dataframe
df['Vertical_Principal'] = df['_cluster_fino'].map(
    lambda x: info_clusters[x]['vertical_industrial'].split('/')[0].strip()
    if x in info_clusters else 'Sin cluster'
)
df['Cluster_Fino'] = df['_cluster_fino'].map(
    lambda x: info_clusters[x].get('nombre_corto', f'Cluster {x}')
    if x in info_clusters else 'Sin cluster'
)

# ── Gráfico ───────────────────────────────────────────────────────────
verticales_unicas = [v for v in df['Vertical_Principal'].unique() if v != 'Sin cluster']
colores_base = sns.color_palette('tab20', n_colors=max(len(verticales_unicas), 1))
mapa_color = {v: colores_base[i] for i, v in enumerate(verticales_unicas)}
mapa_color['Sin cluster'] = (0.85, 0.85, 0.85)

fig, ax = plt.subplots(figsize=(16, 12))

# Ruido en gris
mask_ruido = df['Vertical_Principal'] == 'Sin cluster'
ax.scatter(df[mask_ruido]['_umap_x'], df[mask_ruido]['_umap_y'],
           c=[(0.85, 0.85, 0.85)], s=3, alpha=0.3, zorder=1)

# Clusters con color por vertical
for vertical in verticales_unicas:
    mask = df['Vertical_Principal'] == vertical
    ax.scatter(df[mask]['_umap_x'], df[mask]['_umap_y'],
               c=[mapa_color[vertical]], s=8, alpha=0.6,
               label=f'{vertical} ({mask.sum():,})', zorder=2)

# Etiqueta en el centroide de cada cluster fino
for cid, info in info_clusters.items():
    mask = df['_cluster_fino'] == cid
    if mask.sum() == 0:
        continue
    cx = df[mask]['_umap_x'].mean()
    cy = df[mask]['_umap_y'].mean()
    ax.annotate(
        info.get('nombre_corto', f'C{cid}'), (cx, cy),
        fontsize=6, ha='center',
        bbox=dict(boxstyle='round,pad=0.15', facecolor='white', alpha=0.6, edgecolor='none')
    )

nombre_inst_safe = NOMBRE_INSTITUCION.replace(' ', '_')
ax.set_title(
    f'Mapa de Conocimiento — {NOMBRE_INSTITUCION}\nCada punto = 1 paper, color = vertical industrial',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('UMAP dim 1', fontsize=9)
ax.set_ylabel('UMAP dim 2', fontsize=9)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8, frameon=True)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()

nombre_imagen = f'mapa_conocimiento_{nombre_inst_safe}.png'
plt.savefig(nombre_imagen, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Mapa guardado como {nombre_imagen}')

## Celda 9 — Exportar Excel con resultados

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

def hacer_header(ws, color='1F4E79'):
    fill = PatternFill('solid', fgColor=color)
    font = Font(bold=True, color='FFFFFF', name='Arial', size=10)
    aln  = Alignment(horizontal='center', vertical='center', wrap_text=True)
    thin = Side(style='thin', color='BFBFBF')
    brd  = Border(left=thin, right=thin, top=thin, bottom=thin)
    for cell in ws[1]:
        cell.fill = fill; cell.font = font
        cell.alignment = aln; cell.border = brd
    ws.row_dimensions[1].height = 30

def escribir_df(ws, dataframe, alto=30):
    alt_fill = PatternFill('solid', fgColor='DEEAF1')
    font = Font(name='Arial', size=9)
    aln  = Alignment(wrap_text=True, vertical='top')
    thin = Side(style='thin', color='BFBFBF')
    brd  = Border(left=thin, right=thin, top=thin, bottom=thin)
    for c, col in enumerate(dataframe.columns, 1):
        ws.cell(1, c, col)
    hacer_header(ws)
    for r, (_, row) in enumerate(dataframe.iterrows(), 2):
        for c, val in enumerate(row, 1):
            cell = ws.cell(r, c, str(val) if pd.notna(val) else '')
            cell.font = font; cell.border = brd; cell.alignment = aln
            if r % 2 == 0: cell.fill = alt_fill
        ws.row_dimensions[r].height = alto

wb = Workbook()

# ── Hoja 1: Dataset con clusters ──────────────────────────────────────
ws1 = wb.active
ws1.title = 'Dataset Clusterizado'
cols = [COL_TITULO]
if 'Year' in df.columns:         cols.append('Year')
if 'Source Title' in df.columns: cols.append('Source Title')
cols += ['Vertical_Principal', 'Cluster_Fino']
cols = [c for c in cols if c in df.columns]
escribir_df(ws1, df[cols].reset_index(drop=True), alto=35)
for col, w in zip(['A','B','C','D','E'], [55, 8, 35, 40, 35]):
    ws1.column_dimensions[col].width = w

# ── Hoja 2: Diccionario de clusters ───────────────────────────────────
ws2 = wb.create_sheet('Diccionario de Clusters')
filas_dict = []
for cid in sorted(info_clusters.keys()):
    info = info_clusters[cid]
    filas_dict.append({
        'Cluster ID':           cid,
        'Nombre del cluster':   info.get('nombre_corto', f'Cluster {cid} (sin nombre)'),
        'Vertical principal':   info.get('vertical_industrial', '').split('/')[0].strip(),
        'Verticales completas': info.get('vertical_industrial', ''),
        'Papers':               info['n_papers'],
        'Top keywords':         ' | '.join(info['keywords'][:12]),
        'Títulos ejemplo':      ' | '.join(info['titulos_muestra'][:3]),
    })
df_dict = pd.DataFrame(filas_dict)
escribir_df(ws2, df_dict, alto=50)
for col, w in zip(['A','B','C','D','E','F','G'], [10, 38, 30, 45, 10, 65, 90]):
    ws2.column_dimensions[col].width = w

# ── Hoja 3: Resumen por vertical principal ────────────────────────────
ws3 = wb.create_sheet('Resumen Verticales')
resumen = df['Vertical_Principal'].value_counts().reset_index()
resumen.columns = ['Vertical', 'Papers']
resumen['% del total'] = (resumen['Papers'] / len(df) * 100).round(1).astype(str) + '%'
resumen.insert(0, '#', ['#'+str(i+1) for i in range(len(resumen))])
escribir_df(ws3, resumen, alto=40)
for col, w in zip(['A','B','C','D'], [5, 45, 12, 14]):
    ws3.column_dimensions[col].width = w

wb.save(RUTA_SALIDA)
print(f'✅ Excel guardado: {RUTA_SALIDA}')
print(f'   Hoja 1: {len(df):,} papers con Vertical_Principal y Cluster_Fino')
print(f'   Hoja 2: Diccionario de {len(info_clusters)} clusters')
print(f'   Hoja 3: Resumen por vertical principal')

## Celda 10 — Resumen final

In [ ]:
n_clusterizados = (df['Vertical_Principal'] != 'Sin cluster').sum()
n_sin_cluster   = (df['Vertical_Principal'] == 'Sin cluster').sum()

print(f'═══════════════════════════════════════════════')
print(f'  RESUMEN FINAL — {NOMBRE_INSTITUCION}')
print(f'═══════════════════════════════════════════════')
print(f'  Total papers analizados:  {len(df):,}')
print(f'  Papers clusterizados:     {n_clusterizados:,} ({100*n_clusterizados/len(df):.1f}%)')
print(f'  Papers sin cluster:       {n_sin_cluster:,} ({100*n_sin_cluster/len(df):.1f}%)')
print(f'  Clusters finos:           {len(info_clusters)}')
print(f'  Verticales principales:   {df["Vertical_Principal"].nunique() - 1}')
print(f'═══════════════════════════════════════════════')
print(f'\nArchivos generados:')
print(f'  {RUTA_SALIDA}')
print(f'  embeddings_papers.npy')
nombre_inst_safe = NOMBRE_INSTITUCION.replace(" ", "_")
print(f'  mapa_conocimiento_{nombre_inst_safe}.png')